# S12 — Training in Practice

**Week 7 · Mon Oct 5, 2026 · Module 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s12_training_in_practice.ipynb)

Every cell below is a worked example from the [S12 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s12/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s12.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s12.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Overfit one batch first


*Expected output starts with:* `--- correct model ---`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_num_threads(1)   # single-threaded: exact run-to-run reproducibility

def make_model(buggy=False):
    layers = [
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(16 * 4 * 4, 4),
    ]
    if buggy:
        layers.append(nn.Softmax(dim=1))   # BUG: CrossEntropyLoss wants logits
    return nn.Sequential(*layers)

# One batch of 8 random images with random labels -- nothing to generalize,
# everything to memorize. A healthy training setup must drive loss to ~0.
x = torch.randn(8, 1, 16, 16)
y = torch.randint(0, 4, (8,))

for buggy in [False, True]:
    torch.manual_seed(0)
    model = make_model(buggy)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    tag = "buggy (softmax before loss)" if buggy else "correct model"
    print(f"--- {tag} ---")
    for step in range(501):
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        if step % 100 == 0:
            acc = (model(x).argmax(1) == y).float().mean().item()
            print(f"step {step:3d}: loss = {loss.item():.4f}, batch acc = {acc:.2f}")

## The permuted-label test: capacity checks and leak detection


*Expected output starts with:* `true labels:     train acc = 1.0000, fresh val acc = 0.8340`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_num_threads(1)

def make_data(n, rng):
    X = rng.normal(0.0, 0.5, size=(n, 2)).astype(np.float32)
    y = (X[:, 0] * X[:, 1] > 0).astype(np.int64)      # XOR-quadrant task
    X += rng.normal(0.0, 0.1, size=X.shape).astype(np.float32)
    return torch.from_numpy(X), torch.from_numpy(y)

def fit(Xtr, ytr, Xva, yva, steps=4000):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(2, 256), nn.ReLU(),
                          nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        idx = torch.randint(0, len(Xtr), (64,))
        opt.zero_grad()
        loss_fn(model(Xtr[idx]), ytr[idx]).backward()
        opt.step()
    with torch.no_grad():
        tr = (model(Xtr).argmax(1) == ytr).float().mean().item()
        va = (model(Xva).argmax(1) == yva).float().mean().item()
    return tr, va

rng = np.random.default_rng(0)

# Part A: the permuted-label sanity check on a healthy pipeline.
Xtr, ytr = make_data(64, rng)
Xva, yva = make_data(1000, rng)
y_perm = ytr[torch.randperm(len(ytr))]                # labels now meaningless

tr, va = fit(Xtr, ytr, Xva, yva)
print(f"true labels:     train acc = {tr:.4f}, fresh val acc = {va:.4f}")
tr, va = fit(Xtr, y_perm, Xva, yva)
print(f"permuted labels: train acc = {tr:.4f}, fresh val acc = {va:.4f}")

# Part B: the same check EXPOSES a leaky pipeline. Bug: naive 2x
# oversampling applied BEFORE the train/val split, so every example's
# duplicate (with its label) can land in the other split.
X, y = make_data(400, rng)
y_shuf = y[torch.randperm(400)]                       # permutation test labels
Xd, yd = torch.cat([X, X]), torch.cat([y_shuf, y_shuf])   # oversample first (BUG)
split = torch.randperm(800)                                # ...then split
tr_idx, va_idx = split[:400], split[400:]
tr, va = fit(Xd[tr_idx], yd[tr_idx], Xd[va_idx], yd[va_idx])
print(f"leaky pipeline, permuted labels: train acc = {tr:.4f}, "
      f"val acc = {va:.4f}  <- should be ~0.5!")

## Choosing the learning rate from evidence


*Expected output starts with:* `sane learning rate (SGD, lr = 0.1):`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

np.random.seed(0)
torch.manual_seed(0)

# A small but real task: classify which quadrant-pair a noisy blob is in.
def make_data(n, rng):
    X = rng.normal(0.0, 0.5, size=(n, 2)).astype(np.float32)
    y = (X[:, 0] * X[:, 1] > 0).astype(np.int64)      # XOR-like: needs hidden layer
    X += rng.normal(0.0, 0.1, size=X.shape).astype(np.float32)
    return torch.from_numpy(X), torch.from_numpy(y)

def make_model():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                         nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))

def train_sgd(lr, steps, log_every=None):
    rng = np.random.default_rng(0)
    torch.manual_seed(0)
    X, y = make_data(512, rng)
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for step in range(steps):
        idx = torch.randint(0, len(X), (64,))
        opt.zero_grad()
        loss = loss_fn(model(X[idx]), y[idx])
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if log_every and step % log_every == 0:
            print(f"  step {step:3d}: loss = {loss.item():.4f}")
    return losses

print("sane learning rate (SGD, lr = 0.1):")
train_sgd(0.1, 401, log_every=100)
print("too-high learning rate (SGD, lr = 10.0):")
train_sgd(10.0, 401, log_every=100)

# Learning-rate range test: same budget (60 steps) at each candidate lr,
# report the mean loss over the last 10 steps.
print("lr range test (60 SGD steps each):")
for lr in [1e-3, 1e-2, 1e-1, 3e-1, 1.0, 3.0, 10.0]:
    losses = train_sgd(lr, 60)
    tail = np.mean(losses[-10:])
    print(f"  lr = {lr:6.3f}: mean loss of last 10 steps = {tail:.4f}")

## Exploding gradients, and the clip that catches them


*Expected output starts with:* `no clipping (SGD, lr = 0.3):`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)

# A setup prone to exploding gradients: a deep tanh MLP with over-scaled
# initialization, plain SGD on a small regression task.
def make_model():
    torch.manual_seed(0)
    layers = []
    for i in range(8):
        layers += [nn.Linear(32 if i else 1, 32), nn.Tanh()]
    layers += [nn.Linear(32, 1)]
    net = nn.Sequential(*layers)
    with torch.no_grad():
        for p in net.parameters():
            if p.dim() == 2:
                p.mul_(2.0)                 # over-scaled init
    return net

rng = np.random.default_rng(0)
X = torch.from_numpy(rng.uniform(-2, 2, size=(256, 1)).astype(np.float32))
y = torch.sin(3 * X)

def train(clip):
    net = make_model()
    opt = torch.optim.SGD(net.parameters(), lr=0.3)
    loss_fn = nn.MSELoss()
    max_gnorm, first_bad = 0.0, None
    for step in range(801):
        idx = torch.randint(0, len(X), (32,))
        opt.zero_grad()
        loss = loss_fn(net(X[idx]), y[idx])
        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(net.parameters(),
                                               clip if clip else float("inf"))
        if torch.isfinite(gnorm):
            max_gnorm = max(max_gnorm, gnorm.item())
        elif first_bad is None:
            first_bad = step
        opt.step()
        if step % 200 == 0:
            with torch.no_grad():
                full = loss_fn(net(X), y).item()
            print(f"  step {step:3d}: full-data loss = {full:.4f}")
    print(f"  largest finite pre-clip grad norm: {max_gnorm:.2e}"
          + (f"; first non-finite gradient at step {first_bad}" if first_bad
             is not None else ""))

print("no clipping (SGD, lr = 0.3):")
train(clip=None)
print("clip_grad_norm_ to 1.0 (same everything else):")
train(clip=1.0)

## Seeds and their limits, measured


*Expected output starts with:* `seed 0, run 1: 0.8895`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

# XOR-quadrant task: label = 1 if x and y have the same sign.
def make_data(n, rng):
    X = rng.normal(0.0, 0.5, size=(n, 2)).astype(np.float32)
    y = (X[:, 0] * X[:, 1] > 0).astype(np.int64)
    X += rng.normal(0.0, 0.1, size=X.shape).astype(np.float32)
    return torch.from_numpy(X), torch.from_numpy(y)

def run_experiment(seed):
    """One complete training run, fully controlled by `seed`."""
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    Xtr, ytr = make_data(256, rng)
    Xte, yte = make_data(2000, rng)
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                          nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for step in range(400):
        idx = torch.randint(0, len(Xtr), (64,))
        opt.zero_grad()
        loss_fn(model(Xtr[idx]), ytr[idx]).backward()
        opt.step()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

# Same seed twice: bitwise-identical result.
print(f"seed 0, run 1: {run_experiment(0):.4f}")
print(f"seed 0, run 2: {run_experiment(0):.4f}")

# Different seeds: same code, same data distribution, different answers.
accs = [run_experiment(s) for s in range(5)]
print("seeds 0-4:", ", ".join(f"{a:.4f}" for a in accs))
print(f"mean = {np.mean(accs):.4f}, std = {np.std(accs, ddof=1):.4f}, "
      f"spread = {max(accs) - min(accs):.4f}")

## How many seeds? The standard error of the mean


*Expected output starts with:* `accuracies, seeds 0-19 (sorted):`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)

# Same task and training run as the seed study, now across 20 seeds.
def make_data(n, rng):
    X = rng.normal(0.0, 0.5, size=(n, 2)).astype(np.float32)
    y = (X[:, 0] * X[:, 1] > 0).astype(np.int64)
    X += rng.normal(0.0, 0.1, size=X.shape).astype(np.float32)
    return torch.from_numpy(X), torch.from_numpy(y)

def run_experiment(seed):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    Xtr, ytr = make_data(256, rng)
    Xte, yte = make_data(2000, rng)
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                          nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for step in range(400):
        idx = torch.randint(0, len(Xtr), (64,))
        opt.zero_grad()
        loss_fn(model(Xtr[idx]), ytr[idx]).backward()
        opt.step()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

accs = np.array([run_experiment(s) for s in range(20)])
print("accuracies, seeds 0-19 (sorted):")
print("  " + ", ".join(f"{a:.4f}" for a in np.sort(accs)))
print(f"mean = {accs.mean():.4f}, std = {accs.std(ddof=1):.4f}, "
      f"spread = {accs.max() - accs.min():.4f}")
for n in [3, 5, 10, 20]:
    sub = accs[:n]
    sem = sub.std(ddof=1) / np.sqrt(n)
    print(f"first {n:2d} seeds: mean = {sub.mean():.4f}, "
          f"SEM = {sem:.4f}, mean +/- 2*SEM = "
          f"[{sub.mean() - 2 * sem:.4f}, {sub.mean() + 2 * sem:.4f}]")

## Statistical power: what your seed budget can detect


*Expected output starts with:* `seeds  crit power(d=0.005) power(d=0.011) power(d=0.022)`


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Monte Carlo power analysis for A-vs-B comparisons.
# Model: each seed's score ~ N(mean, sigma^2), sigma from the 20-seed study.
sigma = 0.011          # per-seed std of test accuracy (measured above)

def detect(a, b, crit):
    """Detection statistic: |mean difference| / standard error of it."""
    se = np.sqrt(a.var(ddof=1, axis=1) / a.shape[1]
                 + b.var(ddof=1, axis=1) / b.shape[1])
    return np.abs(a.mean(axis=1) - b.mean(axis=1)) / se > crit

reps = 100_000
print(f"{'seeds':>5} {'crit':>5} " +
      " ".join(f"power(d={d:.3f})" for d in (0.005, 0.011, 0.022)))
for n in [3, 5, 10, 20]:
    # Calibrate the 5% false-positive threshold under the null (no difference),
    # instead of trusting small-sample formulas.
    a = rng.normal(0.0, sigma, size=(reps, n))
    b = rng.normal(0.0, sigma, size=(reps, n))
    se = np.sqrt(a.var(ddof=1, axis=1) / n + b.var(ddof=1, axis=1) / n)
    crit = np.quantile(np.abs(a.mean(axis=1) - b.mean(axis=1)) / se, 0.95)
    powers = []
    for delta in [0.005, 0.011, 0.022]:      # true gaps: 0.5, 1, 2 sigma
        a = rng.normal(0.0, sigma, size=(reps, n))
        b = rng.normal(delta, sigma, size=(reps, n))
        powers.append(detect(a, b, crit).mean())
    print(f"{n:5d} {crit:5.2f} " + " ".join(f"{p:13.3f}" for p in powers))

## Ablations: earning your explanations


*Expected output starts with:* `baseline        acc = 0.8762 +/- 0.0118 (seeds: 0.889, 0.872, 0.867)`


In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn

def make_data(n, rng):
    X = rng.normal(0.0, 0.5, size=(n, 2)).astype(np.float32)
    y = (X[:, 0] * X[:, 1] > 0).astype(np.int64)
    X += rng.normal(0.0, 0.1, size=X.shape).astype(np.float32)
    return torch.from_numpy(X), torch.from_numpy(y)

def build_model(width, depth):
    layers, d_in = [], 2
    for _ in range(depth):
        layers += [nn.Linear(d_in, width), nn.ReLU()]
        d_in = width
    layers.append(nn.Linear(d_in, 2))
    return nn.Sequential(*layers)

def run(config, seed):
    """Every choice lives in `config`; every run is a (config, seed) pair."""
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    Xtr, ytr = make_data(256, rng)
    Xte, yte = make_data(2000, rng)
    model = build_model(config["width"], config["depth"])
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.CrossEntropyLoss()
    for step in range(config["steps"]):        # equal budget for every variant
        idx = torch.randint(0, len(Xtr), (config["batch_size"],))
        opt.zero_grad()
        loss_fn(model(Xtr[idx]), ytr[idx]).backward()
        opt.step()
    with torch.no_grad():
        acc = (model(Xte).argmax(1) == yte).float().mean().item()
    return {"config": config, "seed": seed, "test_acc": round(acc, 4)}

base = {"width": 32, "depth": 2, "lr": 1e-3, "steps": 400, "batch_size": 64}
variants = {
    "baseline":       dict(base),
    "half width":     dict(base, width=16),
    "no hidden depth": dict(base, depth=1),
}

log_path = "ablation_log.jsonl"
with open(log_path, "w") as f:
    for name, config in variants.items():
        accs = []
        for seed in range(3):                  # same 3 seeds for every variant
            record = run(config, seed) | {"variant": name}
            f.write(json.dumps(record) + "\n")
            accs.append(record["test_acc"])
        print(f"{name:15s} acc = {np.mean(accs):.4f} +/- {np.std(accs, ddof=1):.4f} "
              f"(seeds: {', '.join(f'{a:.3f}' for a in accs)})")

with open(log_path) as f:
    first = f.readline().strip()
print("first log line:")
print(first)

## Try it yourself

1. Extend the overfit-one-batch script with two more injected bugs: (a) shuffle `y` every step, (b) construct the optimizer as `torch.optim.Adam([], lr=1e-3)` over an empty parameter list (wrap in `try/except` if needed) or freeze all parameters. Record the signature each leaves on the loss curve.
2. Reproduce the range test with Adam instead of SGD. Where does the usable band sit now, and by how many decades did it move?
3. Take the transfer-learning experiment from [Section 2.3]({{ '/readings/ch2/transfer-learning/' | relative_url }}) and deliberately overfit it: train the from-scratch model to 100% training accuracy while logging validation accuracy each epoch. Identify the epoch where early stopping should have triggered.
4. Write a ten-line `sanity_check(model, loss_fn, x, y, num_classes)` function that asserts (a) initial loss near `ln(num_classes)` and (b) loss below 0.01 after 300 Adam steps on one batch. Keep it; it is reusable in HW3 and your project.


---

Full discussion of everything above: [S12 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s12/).
